In [88]:
!pip install numpy==1.23.5
!pip install scikit-surprise==1.1.3

In [89]:
from surprise import Dataset, Reader
from surprise import SVD, KNNWithMeans
from surprise.model_selection import train_test_split
from surprise.model_selection import GridSearchCV
from surprise import accuracy

In [90]:
import kagglehub
import os
import pandas as pd

path = kagglehub.dataset_download("ayushimishra2809/movielens-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/movielens-dataset


#Carga de datos, modelo de MovieLens 100k

In [91]:
df_ratings = pd.read_csv(os.path.join(path, "ratings.csv"))
df_ratings.head()
df_movies = pd.read_csv(os.path.join(path, "movies.csv"))
df_movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [92]:
len(df_ratings['movieId'].unique())

10325

#Creación de dataframe con las peliculas no rateadas de cada usuario

In [93]:
user_movie_matrix = df_ratings.pivot_table(index='userId', columns='movieId', values='rating')
user_movie_matrix.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,144482,144656,144976,146344,146656,146684,146878,148238,148626,149532
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5.0,NaN,2.0,NaN,3.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,3.0,NaN,3.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [94]:
df_largo = (
    user_movie_matrix
        .reset_index()                  # convierte el índice (userId) en columna
        .melt(id_vars='userId',         # columna que se mantiene fija
              var_name='movieId',       # nombre de la columna “película”
              value_name='rating')      # nombre de la columna “nota”)
)

df_largo.head()

,userId,movieId,rating
0,1,1,NaN
1,2,1,5.0
2,3,1,NaN
3,4,1,NaN
4,5,1,4.0


In [95]:
df_largo[df_largo['rating'].isna()]

,userId,movieId,rating
0,1,1,NaN
2,3,1,NaN
3,4,1,NaN
5,6,1,NaN
6,7,1,NaN
...,...,...,...
6897095,664,149532,NaN
6897096,665,149532,NaN
6897097,666,149532,NaN
6897098,667,149532,NaN


#Preparación de datos para crear el modelo SVD()

In [96]:
df_ratings.drop(columns=['timestamp'], inplace=True)

In [97]:
data = Dataset.load_from_df(df_ratings, Reader(rating_scale=(0.5, 5)))

trainset = data.build_full_trainset()

In [98]:
# trainset es el objeto retornado por algo.fit(trainset) o data.build_full_trainset()
all_inner_users = trainset.all_users()  # genera índices internos (ints)

# Para ver los IDs "reales" (raw ids)
all_raw_users = [trainset.to_raw_uid(inner_id) for inner_id in all_inner_users]

print(all_raw_users)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 22

In [99]:
trainset2, testset2 = train_test_split(data, test_size=0.2, random_state=67)

param_grid = {
    'n_factors': [50, 100, 150],         # número de factores latentes
    'lr_all': [0.01, 0.002, 0.005],       # tasa de aprendizaje
    'reg_all': [0.02, 0.1]          # regularización
}

gs = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3, n_jobs=-1, joblib_verbose=1)
gs.fit(data)

print("Mejor RMSE:", gs.best_score['rmse'])
print("Mejores hiperparámetros:", gs.best_params['rmse'])



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:  1.1min


Mejor RMSE: 0.862076118937276
Mejores hiperparámetros: {'n_factors': 150, 'lr_all': 0.01, 'reg_all': 0.1}


[Parallel(n_jobs=-1)]: Done  54 out of  54 | elapsed:  1.3min finished


In [100]:
# Entrenar SVD
model = SVD(n_factors=150,lr_all=0.01, reg_all=0.1,random_state=67)
model.fit(trainset2)

In [101]:
# Obtener películas vistas por el usuario
uid = 329  # el ID real (raw ID) del usuario que te interesa
seen_items = set([j for (j, _) in trainset2.ur[trainset2.to_inner_uid(uid)]])

In [102]:
# Lista completa de IDs de películas en trainset
all_items = set(trainset2.all_items())
for movieid in df_ratings[df_ratings['userId'] == int('7')]:
  seen_items.add(movieid)
unseen_items = all_items - seen_items
# Convertir a IDs "reales"
unseen_movie_ids = [trainset2.to_raw_iid(iid) for iid in unseen_items]

# Predecir la puntuación para cada película no vista
predictions = [model.predict(uid, iid) for iid in unseen_movie_ids]

predictions = model.test(testset2)
print("RMSE:", accuracy.rmse(predictions))

# Ordenar por predicción más alta
top_preds = sorted(predictions, key=lambda x: x.est, reverse=True)[:50]

RMSE: 0.8543
RMSE: 0.8542965417986412


#Rellenar el dataframe de peliculas sin rating con los ratings predichos

In [103]:
from tqdm import tqdm

# Si no lo está, asegúrate de que userId y movieId sean enteros
df_largo['userId'] = df_largo['userId'].astype(int)
df_largo['movieId'] = df_largo['movieId'].astype(int)

# Predecir valores usando el modelo entrenado
df_largo['rating'] = [
    model.predict(uid=row.userId, iid=row.movieId).est
    for _, row in tqdm(df_largo.iterrows(), total=len(df_largo))
]

100%|██████████| 6897100/6897100 [08:35<00:00, 13387.04it/s]


In [104]:
df_largo.head(10)

,userId,movieId,rating
0,1,1,3.574880
1,2,1,4.101095
2,3,1,3.876234
3,4,1,4.026479
4,5,1,3.484776
5,6,1,4.170152
6,7,1,3.779770
7,8,1,4.143579
8,9,1,3.121101
9,10,1,3.682341


#Ver ratings de los usuarios

In [105]:
usuario = 1
top10_usuario = (
    df_largo[df_largo['userId'] == usuario]
    .sort_values(by='rating', ascending=False)
    .head(10)
)

In [106]:
# Unir con el DataFrame de películas
top10_usuario = top10_usuario.merge(df_movies[['movieId', 'title']], on='movieId', how='left')

# Ordenar columnas si quieres
top10_usuario = top10_usuario[['movieId', 'title', 'rating']]

In [107]:
top10_usuario

,movieId,title,rating
0,1178,Paths of Glory (1957),4.383265
1,1217,Ran (1985),4.362930
2,1248,Touch of Evil (1958),4.337235
3,48516,"Departed, The (2006)",4.265073
4,1172,Cinema Paradiso (Nuovo cinema Paradiso) (1989),4.254703
5,5008,Witness for the Prosecution (1957),4.253913
6,923,Citizen Kane (1941),4.250656
7,318,"Shawshank Redemption, The (1994)",4.228363
8,3000,Princess Mononoke (Mononoke-hime) (1997),4.225855
9,94466,Black Mirror (2011),4.224540
